# Run Simulation

Train Stacked XGBoost models then simulate a season based on it.

## Prepare Data

In [1]:
import pandas as pd
import numpy as np
import os
import statsmodels.api as sm
import plotly.express as px
import xgboost as xgb

print(os.getcwd())
os.chdir('..')
from analytics import RaceData

pd.set_option('future.no_silent_downcasting', True)

C:\Projects\Indycar\repos\indycar-analytics\notebooks


In [2]:
# load results data
rd = RaceData()
rd.add_races_by_date('2015-01-01','2025-12-31',section_results=False)
df = rd.results_df.copy()
df['Rookie'] = df.Driver.str.match(r'.+\(R\)').astype(int)
df['DRIVER'] = df['Driver'].str.replace(' (R)','').str.upper()
df['Engine'] = df['C/A/E/T'].str.extract(r'D\/[\-CH]{1}\/([CH])\/F') # engine

In [3]:
# join track types
rt = pd.read_csv('./cleandata/other/Tracks.csv')
rt['RaceID'] = rt.RaceID.astype(str)
rt['Date'] = pd.to_datetime(rt['Date'])
rt['Year'] = rt['Date'].dt.year
dft = df.merge(rt,on='RaceID')

In [4]:
from datetime import datetime
dft['AeroKit'] = dft['Date'].apply(lambda x: 1 if x < datetime(2018,1,1) else 0)
dft['AeroScreen'] = dft['Date'].apply(lambda x: 1 if x > datetime(2020,1,1) else 0)
dft['Hybrid'] = dft['Date'].apply(lambda x: 1 if x > datetime(2024,7,1) else 0)

In [5]:
teams = pd.read_csv('./cleandata/other/Teams.csv')
dftt = dft.merge(teams,on = ['DRIVER','Car'])
dftt.loc[(dftt.Year < dftt.FirstYear) | (dftt.Year > dftt.LastYear),'Team'] = np.nan

In [6]:
dv = pd.read_csv('./cleandata/other/Drivers.csv')
dfttv = dftt.merge(dv,on='DRIVER',how='left')
dfttv['Offset'] = dfttv['Add'].fillna(0)

In [7]:
# get experience variables
dfttv['RaceInCareer'] = dfttv.groupby('DRIVER')['Date'].rank() + dfttv.Offset
dfttv['RaceWithTeam'] = dfttv.groupby(['DRIVER','Team'])['Date'].rank()
dfttv['RaceInSeason'] = dfttv.groupby(['DRIVER','Year'])['Date'].rank()

In [8]:
dfttv['DriverN'] = dfttv.groupby('DRIVER')['RaceID'].transform('count')
dfttv['TrackN'] = dfttv.groupby('Track')['RaceID'].transform('count')
dfttv['TeamN'] = dfttv.groupby('Team')['RaceID'].transform('count')

In [9]:
dfttv['Finished'] = (dfttv['Running / Reason Out'] == 'Running').astype(int)

In [10]:
train_cols = [ 'Rookie', 'DRIVER', 'Engine', 'Type', 'Track', 'Length', 'Year',
       'Team', 'RaceInCareer', 'RaceWithTeam', 'RaceInSeason', 'AeroKit','AeroScreen','Hybrid']

## Train Models

In [11]:
dat = dfttv.copy()
dat.loc[dat.DriverN < 15,'DRIVER'] = 'X'
dat.loc[dat.TeamN < 10,'Team'] = 'X'
dat.loc[dat.TrackN < 30,'Track'] = 'X'

# SP
X = pd.get_dummies(dat[train_cols],drop_first=False)   
y = dat['SP']

In [12]:
train_size = .8
train = X.sample(int(round(len(X.index)*train_size,0)),random_state=42).index
test = X.index[~X.index.isin(train)]

print(len(train)/(len(test)+len(train)))

0.8000881639850121


In [13]:
dtrain_sp = xgb.DMatrix(X.loc[train,:], label=y[train])
dtest_sp = xgb.DMatrix(X.loc[test,:], label=y[test])

sp_params = {'max_depth': 9
                  ,'min_child_weight': 1
                  ,'gamma': 0
                  ,'eta': .01 
                  ,'subsample': .75
                  ,'colsample_bytree': .3
                  ,'objective': 'reg:squarederror'
                  ,'tree_method': 'exact'
                  ,'seed': 42
                 }

early_stop = xgb.callback.EarlyStopping(
    rounds=250,
    metric_name='rmse',   # match your eval_metric
    data_name='test',     # which eval set to monitor (last one by default)
    save_best=True        # <-- this is the key argument that saves the best model
)

# train the model 
models = {}
models['SP'] = xgb.train(
    params=sp_params,
    dtrain=dtrain_sp,
    num_boost_round=100000,
    evals=[(dtrain_sp, 'train'), (dtest_sp, 'test')],
    verbose_eval=100,
    callbacks=[early_stop]   # pass here instead of early_stopping_rounds
)

[0]	train-rmse:7.48330	test-rmse:7.71116
[100]	train-rmse:6.07947	test-rmse:6.67234
[200]	train-rmse:5.37106	test-rmse:6.26162
[300]	train-rmse:4.94029	test-rmse:6.06044
[400]	train-rmse:4.64368	test-rmse:5.96214
[500]	train-rmse:4.43207	test-rmse:5.90081
[600]	train-rmse:4.25906	test-rmse:5.86108
[700]	train-rmse:4.11157	test-rmse:5.83042
[800]	train-rmse:3.95989	test-rmse:5.81686
[900]	train-rmse:3.81627	test-rmse:5.79631
[1000]	train-rmse:3.69060	test-rmse:5.78668
[1100]	train-rmse:3.55664	test-rmse:5.77322
[1200]	train-rmse:3.42379	test-rmse:5.76834
[1300]	train-rmse:3.30154	test-rmse:5.76814
[1400]	train-rmse:3.19959	test-rmse:5.76530
[1500]	train-rmse:3.09079	test-rmse:5.76129
[1600]	train-rmse:2.97678	test-rmse:5.75866
[1700]	train-rmse:2.88645	test-rmse:5.75657
[1800]	train-rmse:2.80111	test-rmse:5.75695
[1900]	train-rmse:2.71328	test-rmse:5.75919
[2000]	train-rmse:2.62601	test-rmse:5.76370
[2001]	train-rmse:2.62549	test-rmse:5.76375


In [14]:
dat = dfttv.copy()
dat.loc[dat.DriverN < 15,'DRIVER'] = 'X'
dat.loc[dat.TeamN < 10,'Team'] = 'X'
dat.loc[dat.TrackN < 30,'Track'] = 'X'

# Finished
Xf = pd.get_dummies(dat[train_cols + ['SP']],drop_first=False)   
yf = dat['Finished']

dtrain_f = xgb.DMatrix(Xf.loc[train,:], label=yf[train])
dtest_f = xgb.DMatrix(Xf.loc[test,:], label=yf[test])

f_params = {'max_depth': 6
              ,'min_child_weight': 5
              ,'gamma': 0
              ,'eta': .01 
              ,'subsample': 1
              ,'colsample_bytree': .1
              ,'objective': 'binary:logistic'
              ,'eval_metric': 'logloss'
              ,'tree_method': 'exact'
              ,'seed': 42
             }

early_stop = xgb.callback.EarlyStopping(
    rounds=250,
    metric_name='logloss',   # match your eval_metric
    data_name='test',     # which eval set to monitor (last one by default)
    save_best=True        # <-- this is the key argument that saves the best model
)

# train the model 
models['Finished'] = xgb.train(
    params=f_params,
    dtrain=dtrain_f,
    num_boost_round=100000,
    evals=[(dtrain_f, 'train'), (dtest_f, 'test')],
    verbose_eval=100,
    callbacks=[early_stop]   # pass here instead of early_stopping_rounds
)

[0]	train-logloss:0.44092	test-logloss:0.41566
[100]	train-logloss:0.42053	test-logloss:0.40430
[200]	train-logloss:0.40827	test-logloss:0.39830
[300]	train-logloss:0.39929	test-logloss:0.39598
[400]	train-logloss:0.39233	test-logloss:0.39540
[500]	train-logloss:0.38710	test-logloss:0.39489
[600]	train-logloss:0.38222	test-logloss:0.39530
[700]	train-logloss:0.37850	test-logloss:0.39600
[757]	train-logloss:0.37635	test-logloss:0.39630


In [15]:
dat = dfttv.copy()
dat.loc[dat.DriverN < 15,'DRIVER'] = 'X'
dat.loc[dat.TeamN < 10,'Team'] = 'X'
dat.loc[dat.TrackN < 30,'Track'] = 'X'

# Pos
Xp = pd.get_dummies(dat[train_cols+['SP','Finished']],drop_first=False)   
yp = dat['Pos']

dtrain_p = xgb.DMatrix(Xp.loc[train,:], label=yp[train])
dtest_p = xgb.DMatrix(Xp.loc[test,:], label=yp[test])

p_params = {'max_depth': 4
              ,'min_child_weight': 10
              ,'gamma': 0
              ,'eta': .01
              ,'subsample': .75
              ,'colsample_bytree': .4
              ,'objective': 'reg:squarederror'
              ,'tree_method': 'exact'
              ,'seed': 42
             }

early_stop = xgb.callback.EarlyStopping(
    rounds=250,
    metric_name='rmse',   # match your eval_metric
    data_name='test',     # which eval set to monitor (last one by default)
    save_best=True        # <-- this is the key argument that saves the best model
)

# train the model 
models['Pos'] = xgb.train(
    params=p_params,
    dtrain=dtrain_p,
    num_boost_round=100000,
    evals=[(dtrain_p, 'train'), (dtest_p, 'test')],
    verbose_eval=100,
    callbacks=[early_stop]   # pass here instead of early_stopping_rounds
)

[0]	train-rmse:7.50304	test-rmse:7.53381
[100]	train-rmse:5.96128	test-rmse:6.06144
[200]	train-rmse:5.32517	test-rmse:5.48844
[300]	train-rmse:5.02319	test-rmse:5.24538
[400]	train-rmse:4.86786	test-rmse:5.13209
[500]	train-rmse:4.77406	test-rmse:5.07675
[600]	train-rmse:4.70696	test-rmse:5.03976
[700]	train-rmse:4.65477	test-rmse:5.01546
[800]	train-rmse:4.61186	test-rmse:4.99641
[900]	train-rmse:4.57671	test-rmse:4.98522
[1000]	train-rmse:4.54551	test-rmse:4.97804
[1100]	train-rmse:4.51722	test-rmse:4.97706
[1200]	train-rmse:4.48830	test-rmse:4.97088
[1300]	train-rmse:4.46063	test-rmse:4.96766
[1400]	train-rmse:4.43516	test-rmse:4.96725
[1500]	train-rmse:4.40972	test-rmse:4.96594
[1600]	train-rmse:4.38404	test-rmse:4.96612
[1700]	train-rmse:4.36038	test-rmse:4.96655
[1800]	train-rmse:4.33723	test-rmse:4.96566
[1895]	train-rmse:4.31550	test-rmse:4.96826


## Set up Simulation

In [16]:
races = pd.read_csv('2026Races.csv')
drivers = pd.read_csv('2026Entries.csv')
cj = races.merge(drivers,how='cross')
cj = cj.loc[(cj.Rounds == 'All') | ((cj.Rounds == 'Indy500') & (cj.Track == 'IMSO'))]

# add race in career and race with team variables
cj['RaceInCareer'] = cj.RIC+cj.RaceInSeason
cj['RaceWithTeam'] = cj.RWT+cj.RaceInSeason
cj.loc[cj.Rounds == 'Indy500','RaceInCareer'] = cj.loc[cj.Rounds == 'Indy500','RIC'] +1
cj.loc[cj.Rounds == 'Indy500','RaceWithTeam'] = cj.loc[cj.Rounds == 'Indy500','RWT'] +1

In [17]:
sim_sp = pd.DataFrame({}, columns=X.columns)
sim_f = pd.DataFrame({}, columns=Xf.columns)
sim_p = pd.DataFrame({}, columns=Xp.columns)

for df in [sim_sp,sim_f,sim_p]:
    for col in ['Rookie', 'Length', 'Year','RaceInCareer','RaceWithTeam', 
                'RaceInSeason', 'AeroKit', 'AeroScreen', 'Hybrid',
                ]:
        df[col] = cj[col]
        
    for i in cj.index:
        drivercol = f'DRIVER_{cj.loc[i,'DRIVER']}'
        enginecol = f'Engine_{cj.loc[i,'Engine']}'
        typecol = f'Type_{cj.loc[i,'Type']}'
        trackcol = f'Track_{cj.loc[i,'Track']}'
        teamcol = f'Team_{cj.loc[i,'Team']}'
        df.loc[i,drivercol] = 1
        df.loc[i,enginecol] = 1
        df.loc[i,teamcol] = 1
        df.loc[i,typecol] = 1
        df.loc[i,trackcol] = 1
    

In [18]:
# set up residual randomizer
imp = {model: models[model].get_score(importance_type='gain') for model in ['SP','Finished','Pos']}
residuals = {
    'SP': y[test] - models['SP'].predict(dtest_sp),
    'Finished': yf[test] - models['Finished'].predict(dtest_f),
    'Pos': yp[test] - models['Pos'].predict(dtest_p)
}

def get_weighted_residuals(DRIVER,Team,Type,model,scale=1.5):

    # pools
    typool = residuals[model].loc[dfttv.loc[test,'Type'] == Type]
    tmpool = residuals[model].loc[dfttv.loc[test,'Team'] == Team]
    dpool = [] if DRIVER == 'X' else residuals[model].loc[dfttv.loc[test,'DRIVER'] == DRIVER]

    # importances
    tyimp = imp[model].get(f'Type_{Type}',0)
    if model == 'Finished': # don't use team and driver weights for Finished model
        dimp = 0
        tmimp = 0        
    else:
        dimp = imp[model].get(f'DRIVER_{DRIVER}',0)
        tmimp = imp[model].get(f'Team_{Team}',0)
        
    return (
        (np.random.choice(dpool) if len(dpool)>0 else 0)*(dimp/(dimp+tmimp+tyimp)) +
        np.random.choice(tmpool)*(tmimp/(dimp+tmimp+tyimp)) +
        np.random.choice(typool)*(tyimp/(dimp+tmimp+tyimp))
    )*scale
    

In [19]:
dfttv['AdjPts'] = dfttv.Pts - dfttv.SP.apply(lambda x: 1 if x == 1 else 0)
dfttv.loc[(dfttv.Track == 'IMSO') & (dfttv.SP == 1),'AdjPts'] -= 11
ptvalues = dfttv.groupby('Pos')['AdjPts'].mean().rename('Pts').reset_index()

In [27]:
import time
import pickle
N_SIMS = 3000

iters = []
start = time.time()
cj['BasePredSP'] = models['SP'].predict(xgb.DMatrix(sim_sp.fillna(0).astype(float)))
for i in range(N_SIMS):
    df_i = cj.copy()

    # Simulate Starting Position
    df_i['SPScore'] = df_i.apply(lambda x: get_weighted_residuals(x['DRIVER'], x['Team'], x['Type'], 'SP'), 
        axis=1
    ) + df_i['BasePredSP']
    df_i['SP'] = df_i.groupby('RaceInSeason')['SPScore'].rank()
    
    # Simulate Finished/DNF using SP sim from previous step
    sim_f['SP'] = df_i['SP']
    df_i['BasePredF'] = models['Finished'].predict(xgb.DMatrix(sim_f.fillna(0).astype(float)))
    df_i['FScore'] = df_i.apply(lambda x: get_weighted_residuals(x['DRIVER'], x['Team'], x['Type'], 'Finished'), 
                                    axis=1
                                ) + df_i['BasePredF']
    df_i['Finished'] = (df_i.FScore > np.random.rand(len(df_i.index))).astype(int)

    # Simulate Position using Finished and SP
    sim_p['SP'] = df_i['SP']
    sim_p['Finished'] = df_i['Finished']
    df_i['BasePredP'] = models['Pos'].predict(xgb.DMatrix(sim_p.fillna(0).astype(float)))
    df_i['PScore'] = df_i.apply(lambda x: get_weighted_residuals(x['DRIVER'], x['Team'], x['Type'], 'Pos'), 
                                    axis=1
                                ) + df_i['BasePredP'] 
    df_i['PScore2'] = df_i['PScore'] + (1-df_i['Finished'])*30 # inflate DNF p scores to force to back
    df_i['Pos'] = df_i.groupby('RaceInSeason')['PScore2'].rank()

    # add points values
    df_i = df_i.merge(ptvalues,on='Pos',how='inner')
    df_i.loc[df_i.SP==1,'Pts'] +=1 # 1 point for pole
    df_i.loc[(df_i.Track == 'IMSO') & (df_i.SP==1),'Pts'] += 11 #12 points for Indy 500 pole
    
    iters.append(df_i[['Driver','RaceInSeason','Type','SP','Finished','Pos','Pts']])

    if i%500 == 499:
        print(f'Reached Iteration {i}')
        with open(f'iters_{0}-{i}_run2.pickle', 'wb') as outfile:
            pickle.dump(iters,outfile)

print(f'{N_SIMS} iterations took {time.time() - start} seconds')      


Reached Iteration 499
Reached Iteration 999
Reached Iteration 1499
Reached Iteration 1999
Reached Iteration 2499
Reached Iteration 2999
3000 iterations took 2821.7684531211853 seconds


In [28]:
ptsdfs = []
stddfs = []
for i,df in enumerate(iters):
    stand = df.groupby('Driver')['Pts'].sum().rename(f'Pts{i}').to_frame()
    stand[f'Rank{i}'] = stand[f'Pts{i}'].rank(ascending=False)
    ptsdfs.append(stand[f'Pts{i}'])
    stddfs.append(stand[f'Rank{i}'])

pd.concat(stddfs,axis=1).agg(['mean','min','max','std'],axis=1).sort_values('mean')

,mean,min,max,std
Driver,,,,
"Palou, Alex",1.496667,1.0,9.0,0.926431
"O'Ward, Pato",2.371333,1.0,12.0,1.413074
"Lundgaard, Christian",4.835333,1.0,18.0,2.570426
"McLaughlin, Scott",5.115333,1.0,21.0,3.023091
"Dixon, Scott",6.495667,1.0,19.0,3.013250
"Malukas, David",7.571000,1.0,23.0,3.905728
"Kirkwood, Kyle",8.012667,1.0,22.0,3.615273
"Newgarden, Josef",8.445667,1.0,22.0,3.978064
"Armstrong, Marcus",9.685000,1.0,21.0,3.866017


In [29]:
N_SIMS = 2500

iters2 = []
start = time.time()
for i in range(N_SIMS):
    df_i = cj.copy()

    # add comps
    df_i.loc[df_i.Driver == 'Collet, Caio','DRIVER'] = 'ABEL, JACOB'
    df_i.loc[df_i.Driver == 'Hauger, Denis','DRIVER'] = 'FOSTER, LOUIS'
    df_i.loc[df_i.Driver == 'Schumacher, Mick','DRIVER'] = 'ERICSSON, MARCUS'
    
    # Simulate Starting Position
    df_i['SPScore'] = df_i.apply(lambda x: get_weighted_residuals(x['DRIVER'], x['Team'], x['Type'], 'SP'), 
        axis=1
    ) + df_i['BasePredSP']
    df_i['SP'] = df_i.groupby('RaceInSeason')['SPScore'].rank()
    
    # Simulate Finished/DNF using SP sim from previous step
    sim_f['SP'] = df_i['SP']
    df_i['BasePredF'] = models['Finished'].predict(xgb.DMatrix(sim_f.fillna(0).astype(float)))
    df_i['FScore'] = df_i.apply(lambda x: get_weighted_residuals(x['DRIVER'], x['Team'], x['Type'], 'Finished'), 
                                    axis=1
                                ) + df_i['BasePredF']
    df_i['Finished'] = (df_i.FScore > np.random.rand(len(df_i.index))).astype(int)

    # Simulate Position using Finished and SP
    sim_p['SP'] = df_i['SP']
    sim_p['Finished'] = df_i['Finished']
    df_i['BasePredP'] = models['Pos'].predict(xgb.DMatrix(sim_p.fillna(0).astype(float)))
    df_i['PScore'] = df_i.apply(lambda x: get_weighted_residuals(x['DRIVER'], x['Team'], x['Type'], 'Pos'), 
                                    axis=1
                                ) + df_i['BasePredP'] 
    df_i['PScore2'] = df_i['PScore'] + (1-df_i['Finished'])*30 # inflate DNF p scores to force to back
    df_i['Pos'] = df_i.groupby('RaceInSeason')['PScore2'].rank()

    # add points values
    df_i = df_i.merge(ptvalues,on='Pos',how='inner')
    df_i.loc[df_i.SP==1,'Pts'] +=1 # 1 point for pole
    df_i.loc[(df_i.Track == 'IMSO') & (df_i.SP==1),'Pts'] += 11 #12 points for Indy 500 pole
    
    iters2.append(df_i[['Driver','RaceInSeason','Type','SP','Finished','Pos','Pts']])

    if i% 100 == 99:
        print(f'Reached Iteration {i}')

    if i%500 == 499:
        with open(f'iters_{0}-{i}_with_comps_run2.pickle', 'wb') as outfile:
            pickle.dump(iters2,outfile)

print(f'{N_SIMS} iterations took {time.time() - start} seconds')      


Reached Iteration 99
Reached Iteration 199
Reached Iteration 299
Reached Iteration 399
Reached Iteration 499
Reached Iteration 599
Reached Iteration 699
Reached Iteration 799
Reached Iteration 899
Reached Iteration 999
Reached Iteration 1099
Reached Iteration 1199
Reached Iteration 1299
Reached Iteration 1399
Reached Iteration 1499
Reached Iteration 1599
Reached Iteration 1699
Reached Iteration 1799
Reached Iteration 1899
Reached Iteration 1999
Reached Iteration 2099
Reached Iteration 2199
Reached Iteration 2299
Reached Iteration 2399
Reached Iteration 2499
2500 iterations took 2396.067870616913 seconds
